In [2]:
import torch
import torch.nn.functional as F

words = open('names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s: i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

print(len(words))

32033


In [3]:
block_size = 3
X, Y = [], []

for w in words[:5]:     # initially lets try with first 5 words
    print(w)
    context = [0] * block_size    # start with [0, 0, 0] --> (3 dot)
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '-->', itos[ix])
        context = context[1:] + [ix]    # slide the window by one step

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... --> e
..e --> m
.em --> m
emm --> a
mma --> .
olivia
... --> o
..o --> l
.ol --> i
oli --> v
liv --> i
ivi --> a
via --> .
ava
... --> a
..a --> v
.av --> a
ava --> .
isabella
... --> i
..i --> s
.is --> a
isa --> b
sab --> e
abe --> l
bel --> l
ell --> a
lla --> .
sophia
... --> s
..s --> o
.so --> p
sop --> h
oph --> i
phi --> a
hia --> .


In [4]:
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

print(X.shape, X.dtype)
print(Y.shape, Y.dtype)

torch.Size([228146, 3]) torch.int64
torch.Size([228146]) torch.int64


In [5]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g)

In [6]:
print(C[5])

tensor([-0.4713,  0.7868])


In [8]:
emb = C[X]
print(emb.shape)

torch.Size([228146, 3, 2])


In [9]:
emb_flat = emb.view(-1,6)
print(emb_flat.shape)

torch.Size([228146, 6])


In [10]:
g = torch.Generator().manual_seed(2147483647)
W1 = torch.randn((6,100), generator=g)
b1 = torch.randn(100, generator=g)

h = torch.tanh(emb_flat @ W1 + b1)
print(h.shape)

torch.Size([228146, 100])


In [11]:
W2 = torch.randn((100,27), generator=g)
b2 = torch.randn(27, generator=g)

logits = h @ W2 + b2
print(logits.shape)

torch.Size([228146, 27])


In [12]:
loss = F.cross_entropy(logits, Y)
print(loss)

tensor(17.9455)


In [13]:
g = torch.Generator().manual_seed(2147483647)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g) * 0.01
b2 = torch.randn(27, generator=g) * 0.01

emb = C[X]
emb_flat = emb.view(-1, 6)
h = torch.tanh(emb_flat @ W1 + b1)
logits = h @ W2 + b2

loss = F.cross_entropy(logits, Y)
print(loss)

tensor(3.3056)


In [16]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g, requires_grad=True)
W1 = torch.randn((6, 100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = (torch.randn((100, 27), generator=g) * 0.01).requires_grad_(True)
b2 = (torch.randn(27, generator=g) * 0.01).requires_grad_(True)

parameters = [C, W1, b1, W2, b2]
print(sum(p.nelement() for p in parameters))

3481


In [17]:
for k in range(200):
    # forward pass
    emb = C[X]
    h = torch.tanh(emb.view(-1,6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Y)

    # backward pass
    for p in parameters: 
        p.grad = None
    loss.backward()

    # update
    for p in parameters:
        p.data += -0.1 * p.grad

    if k % 20 == 0:
        print(f'iter {k}: loss={loss.item():.4f}')

iter 0: loss=3.3032
iter 20: loss=2.8233
iter 40: loss=2.7440
iter 60: loss=2.7100
iter 80: loss=2.6895
iter 100: loss=2.6749
iter 120: loss=2.6634
iter 140: loss=2.6540
iter 160: loss=2.6459
iter 180: loss=2.6387
